# 02 · Event Study (Phase 2)

**Question (H1):** do opportunistic open-market insider purchases, bought at the close the day after the filing, beat randomly timed purchases of the same stocks over 20 and 60 trading days?

**What this notebook does:** coordinates Phase 2 of `project.md`. All logic lives in `src/classify.py`, `src/event_study.py` and `src/edge_stats.py`; this notebook calls them in order and shows each result. `run_study.py` runs the same steps without the explanations.

**Pass/fail bar (Section 4, fixed):** at 20 **or** 60 days, using **market-adjusted** returns (Section 12): edge ≥ +0.5pp, month-clustered t ≥ 2.0, positive edge in both halves, n ≥ 300.

**Outputs** (`results/phase2_event_study/`):
| File | Contents |
|---|---|
| `dedup_counts.csv` | events left after each duplicate-removal step |
| `events_by_year_cohort.csv` | events per filing year × cohort |
| `edge_table.csv` | sample × cohort × horizon × return type: n, mean event, mean control, edge, t, p, half edges |
| `secondary_tests.csv` | H3–H7: tagged vs untagged edge difference, t, half differences, is_finding |
| `h1_criteria.csv` | each H1 criterion at 20 and 60 days, pass/fail |
| `robustness_checks.csv` | median, trimmed edge, top-10-company share (diagnostics only) |
| `../phase2_h8_tech/*.csv` | H8 counts, edge table, breakdowns, robustness, criteria |

## Step 0 · Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from classify import add_cohort_labels, add_filing_tags, build_trade_history
from edge_stats import h1_verdict
from event_study import add_returns_and_price_tags, build_edge_tables, load_market_history, remove_duplicate_events
from h8_tech import H8_SAMPLE_NAME, build_h8_tables, select_h8_events
from load_filings import list_quarters
from load_prices import find_bad_print_tickers
from utils import PROCESSED_DIR, save_results_csv

RESULTS_FOLDER = "phase2_event_study"
H8_FOLDER = "phase2_h8_tech"
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

## Step 1 · Load Phase 1 purchases and the market (SPY) history
SPY's trading days are the trading calendar for everything below.

In [ ]:
purchases = pd.read_parquet(PROCESSED_DIR / "purchases.parquet")
market_history = load_market_history()
trading_calendar = market_history["date"]
print(f"{len(purchases):,} purchases; trading calendar {trading_calendar.min().date()} to {trading_calendar.max().date()}")

## Step 2 · Remove duplicate events
1. Same company, trade date, shares and price → the same trade reported by related filers (e.g. affiliated funds). Keep the earliest.
2. Same insider, same company, filings within 5 trading days → keep the first (Section 6.1).
3. Bad-print screen (v3): drop tickers whose Yahoo history has a one-day 4x-up / 75%-down spike that snaps back within 5 trading days.

In [ ]:
events, dedup_log = remove_duplicate_events(purchases, trading_calendar)
bad_print_tickers = find_bad_print_tickers(events["ticker"].unique())
events = events[~events["ticker"].isin(bad_print_tickers)].reset_index(drop=True)
dedup_log.loc[len(dedup_log)] = {"step": f"bad-print tickers removed ({len(bad_print_tickers)} tickers)", "events": len(events)}
save_results_csv(dedup_log, RESULTS_FOLDER, "dedup_counts.csv")
dedup_log

## Step 3 · Classify insiders: routine / opportunistic / unclassified
Rule confirmed against Cohen, Malloy & Pomorski (NBER w16454, pp. 12–13). At the start of each year, an insider (per company) who made an open-market purchase **or sale** in each of the three preceding years is **routine** if they traded in the same calendar month in all three, otherwise **opportunistic**. Everyone else is **unclassified**.

The trade history is rebuilt from the cached SEC files (all code P and S trades, no size filters) and cached to `data/processed/trade_history.parquet`. The first build takes several minutes.

In [ ]:
trade_history = build_trade_history(list_quarters())
events = add_cohort_labels(events, trade_history)
print(f"{len(trade_history):,} insider trade-days in the history")
events["cohort"].value_counts()

## Step 4 · Filing-based tags (H3, H4, H5a, H5b)
- **H3:** officer title names a CEO or CFO.
- **H4:** shares bought ≥ 10% of shares held before (a first-ever position counts as large).
- **H5a:** ≥ 2 distinct insiders at the company with trade dates within 2 trading days. **H5b:** ≥ 3 within 10 calendar days. Only filings already public by the event's filing date count (no lookahead).

In [ ]:
events = add_filing_tags(events, purchases, trading_calendar)
events[["is_ceo_or_cfo", "is_large_purchase", "is_cluster_2day", "is_cluster_10day"]].mean()

## Step 5 · Returns and price-based tags (H6, H7)
- **Entry** = adjusted close on the first trading day after the filing date.
- **Returns** at 5, 20, 60 and 120 trading days: raw, and market-adjusted (minus SPY over the same days). A horizon past the end of the price series is left blank.
- **H6:** Z(20) ≤ −1.2 on the filing date. **H7:** 20-day average dollar volume, split into terciles within each filing year.

In [ ]:
events = add_returns_and_price_tags(events, market_history)
events.to_parquet(PROCESSED_DIR / "events.parquet", index=False)

events_by_year_cohort = pd.crosstab(events["filing_date"].dt.year, events["cohort"]).reset_index()
events_by_year_cohort = events_by_year_cohort.rename(columns={"filing_date": "filing_year"})
save_results_csv(events_by_year_cohort, RESULTS_FOLDER, "events_by_year_cohort.csv")
events_by_year_cohort

## Step 6 · Random-date control and edge tables
For each event, 200 random entry days from the same ticker's history. **Edge** = mean event return − mean control return. **t** = month-clustered. **p** = share of the 200 draws whose control mean ≥ the event mean. This step takes a while.

In [ ]:
edge_table, secondary_table, robustness_table = build_edge_tables(events, market_history)
save_results_csv(edge_table, RESULTS_FOLDER, "edge_table.csv")
save_results_csv(secondary_table, RESULTS_FOLDER, "secondary_tests.csv")
save_results_csv(robustness_table, RESULTS_FOLDER, "robustness_checks.csv")

## Step 7 · H1 verdict
Full sample, opportunistic cohort, market-adjusted returns, 20 and 60 days.

In [ ]:
verdict, h1_criteria = h1_verdict(edge_table)
save_results_csv(h1_criteria, RESULTS_FOLDER, "h1_criteria.csv")
print(f"H1 verdict: {verdict}")
h1_criteria

## Step 8 · Edge table: market-adjusted (decides H1), full sample

In [ ]:
edge_table[(edge_table["return_type"] == "market_adjusted") & (edge_table["sample"] == "full")]

## Step 9 · Edge table: raw returns, full sample (reported alongside)

In [ ]:
edge_table[(edge_table["return_type"] == "raw") & (edge_table["sample"] == "full")]

## Step 10 · Post-publication sample (2013 onward, exploratory)
Cohen, Malloy & Pomorski was published in 2012. This cannot rescue a failed H1.

In [ ]:
edge_table[edge_table["sample"] == "2013_onward"]

## Step 11 · Secondary tests (H3–H7)
A finding needs t ≥ 2.5 **and** a positive tagged-minus-untagged difference in both halves.

In [ ]:
secondary_table[secondary_table["return_type"] == "market_adjusted"]

## Step 12 · Robustness diagnostics (never decide a verdict)
Median, share of events beating their control, 1%-trimmed edge, and how much of the edge the top 10 companies supply.

In [ ]:
robustness_table[robustness_table["return_type"] == "market_adjusted"]

## Step 13 · H8: established tech, run once (project.md v3)
Tech SIC codes (Magnificent 7 included), filed 2013+, as-traded close ≥ $5, ≥ $10M/day dollar volume, ≥ 3 years listed. Signal: all insider purchases. Control days must pass the same rules. Market-adjusted decides; QQQ-adjusted shown too.

In [ ]:
h8_events, h8_counts = select_h8_events(events)
save_results_csv(h8_counts, H8_FOLDER, "h8_event_counts.csv")
h8_counts

In [ ]:
h8_edge_table, h8_breakdown_table, h8_robustness_table = build_h8_tables(h8_events, market_history)
save_results_csv(h8_edge_table, H8_FOLDER, "h8_edge_table.csv")
save_results_csv(h8_breakdown_table, H8_FOLDER, "h8_breakdown_tests.csv")
save_results_csv(h8_robustness_table, H8_FOLDER, "h8_robustness_checks.csv")

h8_result, h8_criteria = h1_verdict(h8_edge_table, sample=H8_SAMPLE_NAME, cohort="all")
save_results_csv(h8_criteria, H8_FOLDER, "h8_criteria.csv")
print(f"H8 verdict: {h8_result}")
h8_criteria